In [1]:
import matplotlib.pyplot as plt
import numpy as np

# Model parameters
n = 101  # Number of positions
D = 1.0  # Total demand scaling factor
c0 = 0.2  # Cost per pizza
k = 1.0  # Travel cost
p_max = 1.0  # Max price
dp = 0.05  # Price step
e = 0.5  # Demand elasticity parameter (experiment with values like 0.1, 0.5, 0.8)

road = np.linspace(-1, 1, n)
dx = road[1] - road[0]


def calculate_profit(x1, x2, p1, p2, e):
    # Calculate total cost for each customer position j
    tc1 = p1 + k * np.abs(road - x1)
    tc2 = p2 + k * np.abs(road - x2)

    # Elastic demand per customer location j
    D1 = np.maximum(0.0, (D / n) * (1.0 - e * tc1))
    D2 = np.maximum(0.0, (D / n) * (1.0 - e * tc2))

    # Allocation based on lowest cost; ties split customer demand evenly
    mask1 = tc1 < tc2
    mask2 = tc2 < tc1
    mask_tie = np.isclose(tc1, tc2)

    # Calculate total quantity demanded for each firm
    Q1 = np.sum(D1 * mask1) + 0.5 * np.sum(D1 * mask_tie)
    Q2 = np.sum(D2 * mask2) + 0.5 * np.sum(D2 * mask_tie)

    # Calculate profits
    profit_1 = (p1 - c0) * Q1
    profit_2 = (p2 - c0) * Q2

    return profit_1, profit_2


def best_simultaneous_move(x1, x2, p1, p2, firm, e):
    curr_x = x1 if firm == 1 else x2
    curr_p = p1 if firm == 1 else p2
    curr_idx = np.argmin(np.abs(road - curr_x))

    best_x, best_p = curr_x, curr_p
    best_profit = (
        calculate_profit(x1, x2, p1, p2, e)[0]
        if firm == 1
        else calculate_profit(x1, x2, p1, p2, e)[1]
    )

    for dx_idx in [-1, 0, 1]:
        new_idx = curr_idx + dx_idx
        if 0 <= new_idx < len(road):
            cand_x = road[new_idx]

            for dp_val in [-dp, 0, dp]:
                cand_p = curr_p + dp_val

                if 0 <= cand_p <= p_max:
                    if firm == 1:
                        prof = calculate_profit(cand_x, x2, cand_p, p2, e)[0]
                    else:
                        prof = calculate_profit(x1, cand_x, p1, cand_p, e)[1]

                    if prof > best_profit:
                        best_profit = prof
                        best_x, best_p = cand_x, cand_p

    return best_x, best_p


# Experimenting with different values of e
e_values = [0.1, 0.5, 0.8]
results = {}

for e_val in e_values:
    x1, x2 = np.random.choice(road), np.random.choice(road)
    p1, p2 = 0.5, 0.5

    for iteration in range(500):
        old_x1, old_p1 = x1, p1
        old_x2, old_p2 = x2, p2

        x1, p1 = best_simultaneous_move(old_x1, old_x2, old_p1, old_p2, 1, e_val)
        x2, p2 = best_simultaneous_move(old_x1, old_x2, old_p1, old_p2, 2, e_val)

        if (
            np.isclose(x1, old_x1)
            and np.isclose(x2, old_x2)
            and np.isclose(p1, old_p1)
            and np.isclose(p2, old_p2)
        ):
            break

    prof1, prof2 = calculate_profit(x1, x2, p1, p2, e_val)
    results[e_val] = (x1, x2, p1, p2, prof1, prof2)

# Print Summary Table
print(
    f"{'e':<6}| {'x1':<8}| {'x2':<8}| {'p1':<8}| {'p2':<8}| {'Profit 1':<10}| {'Profit 2':<10}"
)
print("-" * 65)
for e_val, res in results.items():
    print(
        f"{e_val:<6}| {res[0]:<8.2f}| {res[1]:<8.2f}| {res[2]:<8.2f}| {res[3]:<8.2f}| {res[4]:<10.3f}| {res[5]:<10.3f}"
    )

e     | x1      | x2      | p1      | p2      | Profit 1  | Profit 2  
-----------------------------------------------------------------
0.1   | 0.00    | 0.00    | 0.25    | 0.25    | 0.023     | 0.023     
0.5   | -0.26   | 0.32    | 0.80    | 0.85    | 0.140     | 0.133     
0.8   | -0.48   | 0.44    | 0.60    | 0.55    | 0.062     | 0.064     
